In [11]:
from contextlib import contextmanager
import datetime as dt
import json
import zipfile

import pandas as pd

import src

In [12]:
CHANNELS = [
    "@spdde",
    "@csumedia",
    "@FDP",
    "@cdutv",
    "@DieGruenen",
    "@AfDFraktionimBundestag",
    "@AfDTV",
    "@DIELINKE",
]

# Channel Metadata

In [13]:
def load_channel_meta(channel):
    path = src.PATH / "data/raw/yt" / channel / "channel_metadata.json"

    with path.open() as f:
        d = json.load(f)

    info = {
        "channel_uploader_id": d["uploader_id"],
        "channel_follower_count": d["channel_follower_count"],
        "channel_collected": dt.datetime.fromtimestamp(d["epoch"]),
        "channel_description": d["description"],
        "channel": d["channel"],
        "channel_id": d["channel_id"],
        "channel_url": d["channel_url"],
    }

    return info


def create_channel_frame(channels):
    results = []
    for channel in channels:
        info = load_channel_meta(channel)
        results.append(info)

    df = pd.DataFrame(results)

    return df


channels = create_channel_frame(CHANNELS)
channels.head()

,channel_uploader_id,channel_follower_count,channel_collected,channel_description,channel,channel_id,channel_url
0,@spdde,31600,2025-02-04 16:15:24,Wir machen soziale Politik für Dich. Abonniere...,SPD,UCSmbK1WtpYn2sOGLvSSXkKw,https://www.youtube.com/channel/UCSmbK1WtpYn2s...
1,@csumedia,6340,2025-02-04 20:55:15,Hallo und herzlich Willkommen auf unserem YouT...,CSU,UC5AagLvRz7ejBrONZVaA13Q,https://www.youtube.com/channel/UC5AagLvRz7ejB...
2,@FDP,26900,2025-02-04 22:58:15,💛 Aus Liebe zur Freiheit\n\nIMPRESSUM \n\nVera...,FDP,UC-sMkrfoQDH-xzMxPNckGFw,https://www.youtube.com/channel/UC-sMkrfoQDH-x...
3,@cdutv,28400,2025-02-05 10:35:07,Die CDU ist die Volkspartei der Mitte. Seit 19...,CDU,UCKyWIEse3u7ExKfAWuDMVnw,https://www.youtube.com/channel/UCKyWIEse3u7Ex...
4,@DieGruenen,32900,2025-02-05 14:30:53,BÜNDNIS 90/DIE GRÜNEN \nPlatz vor dem Neuen To...,BÜNDNIS 90/DIE GRÜNEN,UC7TAA2WYlPfb6eDJCeX4u0w,https://www.youtube.com/channel/UC7TAA2WYlPfb6...


In [14]:
channels.to_parquet(src.PATH / "data/yt_metadata/channels.parquet", compression="gzip")

# Video Metadata

In [25]:
@contextmanager
def load_video_meta_archive(channel):
    path = src.PATH / "data/raw/yt" / channel / "metadata.zip"
    archive = None
    try:
        archive = zipfile.ZipFile(path, "r")
        file_list = archive.infolist()
        yield archive, file_list
    finally:
        if archive is not None:
            archive.close()


def load_video_metadata(channel):
    all_videos = []
    with load_video_meta_archive(channel) as (archive, files):
        for file in files:
            with archive.open(file) as f:
                byte_content = f.read()
                content = byte_content.decode("utf-8")
                content = json.loads(content)
                # assert that file exists
                file_path = src.PATH / "data/raw/yt" / channel / "videos" / f"{content['id']}.m4a"
                try:
                    info = {
                        "video_id": content["id"],
                        "channel": channel,
                        "channel_id": content["channel_id"],
                        "video_title": content["title"],
                        "video_duration": content["duration"],
                        "video_view_count": content["view_count"],
                        "video_like_count": content.get("like_count", None),
                        "video_comment_count": content["comment_count"],
                        "video_was_live": content["is_live"] | content["was_live"],
                        "video_description": content["description"],
                        "video_datetime_upload": dt.datetime.fromtimestamp(content["timestamp"]),
                    }
                except KeyError:
                    print(channel)
                    print(file)
                    raise
                all_videos.append(info)
                if not info["video_was_live"]:
                    assert file_path.is_file(), file_path

    return all_videos


def create_video_frame(channels):
    videos = []
    for channel in channels:
        channel_videos = load_video_metadata(channel)
        videos.extend(channel_videos)

    return pd.DataFrame(videos)

In [26]:
df = create_video_frame(CHANNELS)

In [27]:
df.to_parquet(src.PATH / "data/yt_metadata/videos.parquet", compression="gzip")